# MUNNA BHAI Medical Health System

This notebook contains the Tkinter-based health classification project. Place `Save yourself.csv` in the notebook working directory, and optionally `medical_logo.png`. **Run locally in Jupyter Notebook**: Tkinter requires a graphical desktop and will generally not display in hosted Colab sessions.

*Educational demonstration only; predictions are not medical diagnoses.*

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import tkinter as tk
from tkinter import ttk
from PIL import Image, ImageTk


# Step 1: Load and preprocess data

df = pd.read_csv('Save yourself.csv')
df.dropna(inplace=True)

df.rename(columns={
    "ï»¿Age": "Age",
    "Blood Pressure": "Blood Pressure",
    "Cholesterol Level": "Cholesterol (mg/dL)",
    "Heart Rate": "Heart Rate (bpm)",
    "Respiratory Rate": "Respiratory Rate (breaths/min)",
    "Blood Sugar Level": "Blood Sugar (mg/dL)",
    "Health Status": "status"
}, inplace=True)

# Split BP into systolic/diastolic
df[['Systolic Blood Pressure (mmHg)', 'Diastolic Blood Pressure (mmHg)']] = (
    df['Blood Pressure'].str.split('/', expand=True).astype(int)
)
df.drop(columns=['Blood Pressure'], inplace=True)

# Convert allergens to dummy variables
df = pd.get_dummies(df, columns=['Allergens'], prefix='Allergen')

# Encode categorical columns
encoders = {}
categorical_cols = ["Gender", "status"]
for col in categorical_cols:
    enc = LabelEncoder()
    df[col] = enc.fit_transform(df[col])
    encoders[col] = enc

X = df.drop("status", axis=1)
y = df["status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)


# Step 2: Tkinter UI Setup

root = tk.Tk()
root.title("MUNNA BHAI Medical Health System 🩺")
root.geometry("720x600")
root.config(bg="#E8F0F2")

# Optional logo
try:
    logo_img = Image.open("medical_logo.png").resize((90, 90))
    logo_photo = ImageTk.PhotoImage(logo_img)
except:
    logo_photo = None

# --- Header ---
header_frame = tk.Frame(root, bg="#0A2540")
header_frame.pack(fill="x")

if logo_photo:
    logo_label = tk.Label(header_frame, image=logo_photo, bg="#0A2540")
    logo_label.pack(side="left", padx=15, pady=10)

tk.Label(
    header_frame,
    text="MUNNA BHAI Medical Health System",
    font=("Segoe UI", 20, "bold"),
    bg="#0A2540",
    fg="white"
).pack(side="left", pady=15)

# --- Tabs ---
style = ttk.Style()
style.theme_use("clam")
style.configure("TNotebook.Tab", font=("Segoe UI", 11, "bold"),
                padding=[15, 8], background="#DCE4E8")
style.map("TNotebook.Tab", background=[("selected", "#0A2540")],
          foreground=[("selected", "white")])

notebook = ttk.Notebook(root)
notebook.pack(padx=10, pady=20, fill="both", expand=True)

# --- Tab 1: Patient Info ---
tab1 = tk.Frame(notebook, bg="#F9FAFB")
notebook.add(tab1, text="🧍 Patient Details")

# --- Helper Function ---
def create_label_entry(parent, text, row, input_type="entry", options=None):
    tk.Label(
        parent,
        text=text,
        font=("Segoe UI", 11, "bold"),
        bg="#F9FAFB",
        fg="#0A2540"
    ).grid(row=row, column=0, sticky="w", pady=5, padx=10)

    if input_type == "entry":
        entry = tk.Entry(parent, font=("Segoe UI", 11))
        entry.grid(row=row, column=1, pady=5, padx=10)
    elif input_type == "combo":
        entry = ttk.Combobox(
            parent, values=options or [], font=("Segoe UI", 11), state="readonly"
        )
        entry.grid(row=row, column=1, pady=5, padx=10)
        entry.current(0)
    return entry

# --- Value Ranges for Combos ---
bp_systolic_options = ["Below 80", "80-120", "120-140", "140-160", "160-180", "Above 180"]
bp_diastolic_options = ["Below 60", "60-80", "80-90", "90-100", "100-110", "Above 110"]
cholesterol_options = ["Below 160", "160-200", "200-240", "Above 240"]
heart_rate_options = ["Below 60", "60-80", "80-100", "Above 100"]
resp_rate_options = ["Below 12", "12-16", "16-20", "Above 20"]
blood_sugar_options = ["Below 80", "80-100", "100-125", "Above 125"]

# --- Input Fields ---
age_entry = create_label_entry(tab1, "Age:", 0)
gender_combo = create_label_entry(tab1, "Gender:", 1, "combo", options=["Male", "Female"])
systolic_entry = create_label_entry(tab1, "Systolic BP (mmHg):", 2, "combo", options=bp_systolic_options)
diastolic_entry = create_label_entry(tab1, "Diastolic BP (mmHg):", 3, "combo", options=bp_diastolic_options)
chol_entry = create_label_entry(tab1, "Cholesterol (mg/dL):", 4, "combo", options=cholesterol_options)
heart_entry = create_label_entry(tab1, "Heart Rate (bpm):", 5, "combo", options=heart_rate_options)
resp_entry = create_label_entry(tab1, "Respiratory Rate:", 6, "combo", options=resp_rate_options)
sugar_entry = create_label_entry(tab1, "Blood Sugar (mg/dL):", 7, "combo", options=blood_sugar_options)

# --- Tab 2: Allergies ---
tab2 = tk.Frame(notebook, bg="#F9FAFB")
notebook.add(tab2, text="🌿 Allergies")

tk.Label(
    tab2,
    text="Select Allergies:",
    font=("Segoe UI", 12, "bold"),
    bg="#F9FAFB",
    fg="#0A2540"
).pack(anchor="w", padx=20, pady=10)

allergen_vars = {a: tk.IntVar() for a in ["Dust", "Nuts", "Pollen", "Latex", "None"]}
check_frame = tk.Frame(tab2, bg="#F9FAFB")
check_frame.pack(padx=20, pady=10)

for i, (a, var) in enumerate(allergen_vars.items()):
    tk.Checkbutton(
        check_frame,
        text=a,
        variable=var,
        bg="#F9FAFB",
        font=("Segoe UI", 11)
    ).grid(row=0, column=i, padx=10, pady=10)

# --- Tab 3: Prediction ---
tab3 = tk.Frame(notebook, bg="#F9FAFB")
notebook.add(tab3, text="🩺 Predict")

tk.Label(
    tab3,
    text="Press the button below to analyze the patient’s health status.",
    font=("Segoe UI", 12),
    bg="#F9FAFB",
    fg="#0A2540"
).pack(pady=20)

# Result frame
result_frame = tk.Frame(tab3, bg="#E8F0F2", bd=2, relief="groove")
result_frame.pack(pady=20, padx=40, fill="x")

result_label = tk.Label(
    result_frame,
    text="No result yet.",
    font=("Segoe UI", 12, "italic"),
    bg="#E8F0F2",
    fg="#4A5568",
    justify="center"
)
result_label.pack(pady=10)

# --- Range Conversion Logic ---
def convert_range(value):
    if "Below" in value:
        return float(value.split(" ")[1]) - 1
    elif "Above" in value:
        return float(value.split(" ")[1]) + 1
    elif "-" in value:
        low, high = map(int, value.split("-"))
        return (low + high) / 2
    else:
        return float(value)

# --- Predict Button Logic ---
def predict_status():
    try:
        Age = int(age_entry.get())
        Gender = gender_combo.get()
        Systolic_BP_label = systolic_entry.get()
        Diastolic_BP_label = diastolic_entry.get()
        Cholesterol_label = chol_entry.get()
        Heart_Rate_label = heart_entry.get()
        Respiratory_Rate_label = resp_entry.get()
        Blood_Sugar_label = sugar_entry.get()

        Systolic_BP = convert_range(Systolic_BP_label)
        Diastolic_BP = convert_range(Diastolic_BP_label)
        Cholesterol = convert_range(Cholesterol_label)
        Heart_Rate = convert_range(Heart_Rate_label)
        Respiratory_Rate = convert_range(Respiratory_Rate_label)
        Blood_Sugar = convert_range(Blood_Sugar_label)

        gender_encoded = encoders["Gender"].transform([Gender])[0]
        allergen_input_dict = {f'Allergen_{a}': v.get() for a, v in allergen_vars.items()}
        allergen_sum = sum(v.get() for k, v in allergen_vars.items() if k != "None")

        # --- Custom Logic ---
        all_below = all("Below" in val for val in [
            Systolic_BP_label, Diastolic_BP_label, Cholesterol_label,
            Heart_Rate_label, Respiratory_Rate_label, Blood_Sugar_label
        ])

        if Age > 11 and all_below:
            status = "Unhealthy (Below range for all parameters)"
        else:
            input_data = pd.DataFrame([{
                "Age": Age,
                "Gender": gender_encoded,
                "Systolic Blood Pressure (mmHg)": Systolic_BP,
                "Diastolic Blood Pressure (mmHg)": Diastolic_BP,
                "Cholesterol (mg/dL)": Cholesterol,
                "Heart Rate (bpm)": Heart_Rate,
                "Respiratory Rate (breaths/min)": Respiratory_Rate,
                "Blood Sugar (mg/dL)": Blood_Sugar,
                **allergen_input_dict
            }])
            input_data = input_data.reindex(columns=X_train.columns, fill_value=0)

            proba = model.predict_proba(input_data)[0]
            unhealthy_prob = proba[encoders["status"].transform(["Unhealthy"])[0]]

            # Increase unhealthy chance if allergens present
            if allergen_sum > 0:
                unhealthy_prob += 0.15 * allergen_sum

            status = "Unhealthy" if unhealthy_prob > 0.5 else "Healthy"

        # --- Display result ---
        if "Unhealthy" in status:
            bg_color = "#FED7D7"
            fg_color = "#742A2A"
            emoji = "⚠️"
        else:
            bg_color = "#C6F6D5"
            fg_color = "#22543D"
            emoji = "✅"

        result_frame.config(bg=bg_color)
        result_label.config(
            text=f"{emoji} Predicted Health Status:\n\n{status}",
            bg=bg_color,
            fg=fg_color,
            font=("Segoe UI", 13, "bold")
        )

    except Exception as e:
        result_frame.config(bg="#FFE8B6")
        result_label.config(
            text=f"⚠️ Invalid input or missing field!\n\n{e}",
            fg="#744210",
            bg="#FFE8B6",
            font=("Segoe UI", 11, "italic")
        )

predict_btn = tk.Button(
    tab3,
    text="🔍 Predict Health Status",
    font=("Segoe UI", 14, "bold"),
    bg="#0A2540",
    fg="white",
    activebackground="#1A365D",
    cursor="hand2",
    padx=20,
    pady=10,
    command=predict_status
)
predict_btn.pack(pady=10)

# Footer
footer = tk.Label(
    root,
    text="© 2025 MUNNA BHAI Medical Health System | All Rights Reserved",
    bg="#0A2540",
    fg="white",
    font=("Segoe UI", 9)
)
footer.pack(side="bottom", fill="x")

root.mainloop()